# 10 — Validate first reporting isolate

## Goal

The project uses a patient-deduplicated reporting rule: one earliest record per patient, cohort, base reporting organism, and reporting phenotype.

Because a patient can legitimately contribute both a base organism and a phenotype-specific category, the partition includes `reporting_phenotype`. This is a project reporting policy, not a claim that the source provides a verified isolate identifier.

In [6]:
%sql
with duplicates as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,
        count(*) as records

    from workspace.micro_dev.int_first_reporting_isolate

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype

    having count(*) > 1

)

select
    (
        select count(*)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as selected_rows,

    (
        select count(distinct first_reporting_isolate_key)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as unique_keys,

    (
        select count(*)
        from duplicates
    ) as duplicate_reporting_groups,

    case
        when (
            select count(*)
            from duplicates
        ) = 0

        and (
            select count(*)
            from workspace.micro_dev.int_first_reporting_isolate
        ) = (
            select count(distinct first_reporting_isolate_key)
            from workspace.micro_dev.int_first_reporting_isolate
        )

        then 'pass'
        else 'fail'
    end as validation_status

,selected_rows,unique_keys,duplicate_reporting_groups,validation_status
0,77369,77369,0,pass


### Structural uniqueness

The selection produces 77,369 rows and 77,369 unique keys, with no duplicate patient/cohort/reporting-group/phenotype combinations.

In [1]:
%sql
with expected_ranked as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,

        reporting_culture_organism_key,
        culture_time_shifted,
        culture_key,

        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype

            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as expected_rank

    from workspace.micro_dev.int_reporting_culture_organism

    where culture_time_shifted is not null
),

expected_first as (

    select *
    from expected_ranked
    where expected_rank = 1

),

comparison as (

    select
        e.source_version,
        e.cohort_id,
        e.patient_key,
        e.organism_reporting_group,
        e.reporting_phenotype,

        e.reporting_culture_organism_key
            as expected_reporting_record,

        a.reporting_culture_organism_key
            as actual_reporting_record,

        e.culture_time_shifted
            as expected_time,

        a.culture_time_shifted
            as actual_time

    from expected_first e

    full outer join workspace.micro_dev.int_first_reporting_isolate a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group =
           a.organism_reporting_group
       and e.reporting_phenotype =
           a.reporting_phenotype

)

select
    count(*) as reporting_groups,

    sum(
        case
            when expected_reporting_record
                 = actual_reporting_record
                then 1
            else 0
        end
    ) as correctly_selected,

    sum(
        case
            when expected_reporting_record
                 <> actual_reporting_record
              or expected_reporting_record is null
              or actual_reporting_record is null
                then 1
            else 0
        end
    ) as selection_mismatches,

    case
        when sum(
            case
                when expected_reporting_record
                     <> actual_reporting_record
                  or expected_reporting_record is null
                  or actual_reporting_record is null
                    then 1
                else 0
            end
        ) = 0
            then 'pass'
        else 'fail'
    end as validation_status

from comparison

,reporting_groups,correctly_selected,selection_mismatches,validation_status
0,77369,77369,0,pass


### Earliest-record check

I independently recompute the ranking from `int_reporting_culture_organism` and compare it with the selected model. All 77,369 reporting groups pick the expected earliest record, with zero mismatches.

In [2]:
%sql
select
    resolution_status,
    reconciliation_reason,

    count(*) as endpoint_records,

    sum(
        case
            when source_organisms_with_endpoint > 1
                then 1
            else 0
        end
    ) as multi_source_endpoints

from workspace.micro_dev.int_first_reporting_susceptibility

group by
    resolution_status,
    reconciliation_reason

order by
    resolution_status,
    reconciliation_reason

,resolution_status,reconciliation_reason,endpoint_records,multi_source_endpoints
0,conflict,discordant_source_results,278,278
1,conflict,source_endpoint_conflict,3739,118
2,missing,missing_only,967,1
3,resolved,resolved_consistent_result,981699,1911
4,resolved,resolved_with_missing_source_endpoint,2,2


### Reconcile susceptibility after source-label consolidation

Some reporting records are backed by more than one source culture-organism key. I explode those keys and reconcile each antibiotic again at the reporting-isolate level.

Most endpoints agree cleanly. When two source records disagree, the reporting endpoint stays `conflict`; I do not pick one source row arbitrarily.

In [3]:
%sql
with key_check as (

    select
        count(*) as row_count,
        count(distinct reporting_susceptibility_key) as unique_key_count
    from workspace.micro_dev.int_first_reporting_susceptibility

),

logic_check as (

    select
        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility is null
                    then 1
                else 0
            end
        ) as resolved_with_null_result,

        sum(
            case
                when resolution_status in ('conflict', 'missing')
                 and susceptibility is not null
                    then 1
                else 0
            end
        ) as unresolved_with_result

    from workspace.micro_dev.int_first_reporting_susceptibility

)

select
    k.row_count,
    k.unique_key_count,

    l.resolved_with_null_result,
    l.unresolved_with_result,

    case
        when k.row_count = k.unique_key_count
         and l.resolved_with_null_result = 0
         and l.unresolved_with_result = 0
            then 'pass'
        else 'fail'
    end as validation_status

from key_check k
cross join logic_check l

,row_count,unique_key_count,resolved_with_null_result,unresolved_with_result,validation_status
0,986685,986685,0,0,pass


### Reporting-susceptibility grain and logic

`int_first_reporting_susceptibility` has 986,685 rows and 986,685 unique keys. There are no resolved rows with a null result and no unresolved rows carrying a susceptibility value.

### Takeaway

The patient-level selection and the second-stage susceptibility reconciliation are both deterministic. That gives the marts a stable reporting grain even when multiple source labels were consolidated into one reporting record.